# 00 — IRS Form 990 XML ingestion & fraud-label feasibility (2023–2025)

**Goal:** discover official IRS ZIP archives, download **one archive only**, inspect its XML schema, extract the *Significant Diversion of Assets* disclosure (Form 990 Part VI, Section A, Line 5), and save a reproducible sample audit.

**Important distinctions**
- `publication_year` = year of the IRS download archive; `tax_year` = period reported by the organization. They may differ. This pilot samples 2025 **publication-year** files and audits tax-year coverage (it does not filter the saved pilot).
- `fraud_label=1` means **self-reported significant diversion of assets**, **not confirmed organizational fraud**. `0` means a reported No, **not proof of no fraud**. Missing/ambiguous fields remain null.
- The notebook uses standard-library XML parsing; it does **not** require `irsx`. Form 990-EZ, 990-PF and other forms are excluded from label calculations.
- Downloads are disabled by default (DOWNLOAD_ONE_ARCHIVE = False). Set this to True only when intentionally downloading an IRS archive.

Official source: https://www.irs.gov/charities-non-profits/form-990-series-downloads

## 1 — Imports and project paths
Run from the project folder or a notebook under `notebooks/`. The script finds the project root by looking for `.gitignore`.

In [ ]:
from pathlib import Path
import re, json, time, zipfile, urllib.request
from html.parser import HTMLParser
from collections import Counter
from xml.etree import ElementTree as ET
import pandas as pd

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / ".gitignore").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open JupyterLab from your irs-990-fraud-detection project folder (with .gitignore).")
RAW = ROOT / "data" / "raw"
STAGING = ROOT / "data" / "staging"
REPORTS = ROOT / "reports"
for p in (RAW, STAGING, REPORTS): p.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Pandas:", pd.__version__)

## 2 — Discover the official IRS archive links
The IRS updates its download page; discover ZIP URLs rather than hard-coding a list. **No large downloads in this step.**

In [ ]:
IRS_PAGE = "https://www.irs.gov/charities-non-profits/form-990-series-downloads"
YEARS = {2023, 2024, 2025}

class LinkCollector(HTMLParser):
    def __init__(self):
        super().__init__(); self.links = []
    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            href = dict(attrs).get("href")
            if href: self.links.append(href)

req = urllib.request.Request(IRS_PAGE, headers={"User-Agent": "Mozilla/5.0 IRS990Research/1.0"})
with urllib.request.urlopen(req, timeout=45) as response:
    html = response.read().decode("utf-8", errors="replace")
parser = LinkCollector(); parser.feed(html)
from urllib.parse import urljoin
pattern = re.compile(r"https://apps\.irs\.gov/pub/epostcard/990/xml/(2023|2024|2025)/[^/]+\.zip(?:\?.*)?$", re.I)
archive_urls = sorted({absolute.split("?")[0] for href in parser.links
                       if pattern.match(absolute := urljoin(IRS_PAGE, href))})
archives = pd.DataFrame({"publication_year": [int(pattern.match(u).group(1)) for u in archive_urls], "url": archive_urls})
if archives.empty:
    raise RuntimeError("No matching IRS ZIP links found. Check the IRS page structure and link pattern.")
print(archives.groupby("publication_year").size().rename("archives"))
assert set(archives.publication_year) == YEARS, "Some years are missing: check the IRS page or network access."
display(archives.head(5))

## 3 — Choose **one** archive and download it
By default, use the first 2025 archive. A ZIP may still be hundreds of MB; check available disk space. The download is cached locally and is not committed to Git because `data/raw/` is ignored.

In [ ]:
PILOT_PUBLICATION_YEAR = 2025
year_archives = archives.loc[archives.publication_year.eq(PILOT_PUBLICATION_YEAR), "url"]
if year_archives.empty:
    raise RuntimeError(f"No archive found for publication year {PILOT_PUBLICATION_YEAR}.")
selected = year_archives.iloc[0]
zip_path = RAW / str(PILOT_PUBLICATION_YEAR) / selected.rsplit("/", 1)[-1]
zip_path.parent.mkdir(parents=True, exist_ok=True)
print("Archive:", selected)
print("Local file:", zip_path)
print("Already downloaded:", zip_path.exists())

In [ ]:
# Set to True only when ready to download one potentially large ZIP.
DOWNLOAD_ONE_ARCHIVE = False

if DOWNLOAD_ONE_ARCHIVE:
    if not zip_path.exists():
        temp_path = zip_path.with_suffix(".zip.part")
        req = urllib.request.Request(selected, headers={"User-Agent": "Mozilla/5.0 IRS990Research/1.0"})
        with urllib.request.urlopen(req, timeout=120) as source, temp_path.open("wb") as dest:
            while True:
                block = source.read(1024 * 1024)
                if not block: break
                dest.write(block)
        temp_path.replace(zip_path)
    assert zipfile.is_zipfile(zip_path), "Download is not a valid ZIP. Remove it and retry."
    print(f"Downloaded/verified: {zip_path.stat().st_size / 1024**2:,.1f} MB")
else:
    print("Download skipped. Change DOWNLOAD_ONE_ARCHIVE to True when ready.")

## 4 — Inspect ZIP contents without extracting files
We stream individual XML members directly from the archive. No giant directory of XML files is created.

In [ ]:
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    print("No valid ZIP yet — complete Step 3 first.")
else:
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.infolist() if m.filename.lower().endswith(".xml") and not m.is_dir()]
        print("XML files:", f"{len(members):,}")
        print("First filenames:", [m.filename for m in members[:5]])
        print("Uncompressed XML size (GB):", round(sum(m.file_size for m in members)/1024**3, 2))

## 5 — Discover how the fraud disclosure is represented in XML
**Do not assume the field name is the same across all IRS schema versions.** Scan tags containing `diversion`, then inspect their values and parent contexts. Review the output before approving the label mapping.

In [ ]:
def local_name(tag):
    return tag.rsplit("}", 1)[-1].split(":")[-1]

def sample_diversion_fields(z, members, limit=100):
    found = Counter(); examples = {}
    for member in members[:limit]:
        try:
            with z.open(member) as f: root = ET.parse(f).getroot()
            for element in root.iter():
                tag = local_name(element.tag)
                if "diversion" in tag.lower():
                    found[tag] += 1
                    examples.setdefault(tag, (element.text or "").strip()[:180])
        except ET.ParseError:
            continue
    return pd.DataFrame([{"xml_tag": k, "count_in_sample": v, "example_text": examples[k]} for k, v in found.most_common()])

if zip_path.exists() and zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.infolist() if m.filename.lower().endswith(".xml") and not m.is_dir()]
        discovery = sample_diversion_fields(z, members, limit=min(200, len(members)))
    display(discovery)
    if discovery.empty:
        print("No diversion tags found in this sample. Inspect additional filings/schema before mapping.")

In [ ]:
# Inspect the diversion field and its actual values

if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    print("ZIP archive not found. Download or provide the archive first.")
else:
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.infolist() if m.filename.lower().endswith(".xml") and not m.is_dir()]
        found_count = 0

        for member in members[:500]:
            try:
                with z.open(member) as f:
                    root = ET.parse(f).getroot()

                for element in root.iter():
                    tag = local_name(element.tag)

                    if "diversion" in tag.lower():
                        print(
                            "XML field:", tag,
                            "| Value:", repr(element.text)
                        )
                        found_count += 1

                if found_count >= 10:
                    break

            except (ET.ParseError, OSError, KeyError) as exc:
                print(f"Skipping {member.filename}: {exc}")

## 6 — Parse a bounded pilot sample
The parser only assigns a label when it finds a **recognized Yes/No element**. Unrecognized variants remain missing and are listed in the audit. Only full Form 990 returns are eligible. This is a *pilot*, not the complete three-year dataset.

In [ ]:
def extract_diversion_response(root):
    """
    Extract Form 990 Part VI, Line 5:
    Significant Diversion of Assets.
    """
    for element in root.iter():
        if local_name(element.tag) == "MaterialDiversionOrMisuseInd":
            return element.text.strip() if element.text else None

    return None


def parse_fraud_label(value):
    """
    Convert IRS disclosure responses into binary labels.

    1 = Significant diversion disclosed
    0 = No significant diversion disclosed
    None = Missing or unrecognized
    """
    if value is None:
        return None

    value = str(value).strip().lower()

    if value in ("1", "true", "yes", "y"):
        return 1

    if value in ("0", "false", "no", "n"):
        return 0

    return None

In [ ]:
# Candidate names must be verified against the discovery output above.
# Common IRS e-file schema names include these variants; review any other names before adding.
LABEL_TAGS = {"MaterialDiversionOrMisuseInd"}

MAX_XML_FILES = 5000

def first_text(root, names):
    for el in root.iter():
        if local_name(el.tag) in names and el.text and el.text.strip():
            return el.text.strip()
    return None

def parse_filing(root, filename, publication_year):
    # Find IRS990 form body (not IRS990EZ, IRS990PF, etc.)
    form = next(
        (e for e in root.iter() if local_name(e.tag) == "IRS990"),
        None
    )

    if form is None:
        return None

    # Extract filing header information
    header = next(
        (e for e in root.iter() if local_name(e.tag) == "ReturnHeader"),
        root
    )

    # Extract significant diversion of assets disclosure
    raw_label = first_text(form, LABEL_TAGS)

    # Convert disclosure to binary label using the shared parser
    label = parse_fraud_label(raw_label)

    # Extract organization information
    tax_year = first_text(header, {"TaxYr", "TaxYear"})
    ein = first_text(header, {"EIN"})
    name = first_text(
        header,
        {"BusinessNameLine1Txt", "BusinessNameLine1", "BusinessName"}
    )

    return {
        "publication_year": publication_year,
        "source_xml": filename,
        "ein": ein.zfill(9) if ein and ein.isdigit() else ein,
        "organization_name": name,
        "tax_year": int(tax_year) if tax_year and tax_year.isdigit() else None,
        "raw_diversion_response": raw_label,
        "fraud_label": label,
        "disclosure_field_found": raw_label is not None,
    }

records, errors = [], []
pilot = pd.DataFrame(
    columns=[
        "publication_year",
        "source_xml",
        "ein",
        "organization_name",
        "tax_year",
        "raw_diversion_response",
        "fraud_label",
        "disclosure_field_found",
    ]
)

if zip_path.exists() and zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        xml_members = [m for m in z.infolist() if m.filename.lower().endswith(".xml") and not m.is_dir()]
        for m in xml_members[:MAX_XML_FILES]:
            try:
                with z.open(m) as f: root = ET.parse(f).getroot()
                record = parse_filing(root, m.filename, PILOT_PUBLICATION_YEAR)
                if record is not None: records.append(record)
            except (ET.ParseError, ValueError, OSError) as exc:
                errors.append({"source_xml": m.filename, "error": str(exc)[:300]})
    if records:
        pilot = pd.DataFrame(records)
    print("XML sampled:", min(MAX_XML_FILES, len(xml_members)))
    print("Full Form 990 parsed:", len(pilot))
    print("XML parse errors:", len(errors))
    display(pilot.head())
else:
    print("Provide a valid ZIP archive before running this step.")

## 7 — Audit label completeness and tax-year coverage
**Stop and investigate** if the Yes/No field is missing for nearly every Form 990: the XML tag mapping may need revision. A missing response is **not** a No. Do not train a model from this pilot yet.

In [ ]:
if records:
    print("Label distribution (including missing):")
    display(pilot["fraud_label"].map({1.0:"Yes — disclosed",0.0:"No — not disclosed"}).fillna("Missing/ambiguous").value_counts(dropna=False).rename_axis("status").to_frame("filings"))
    print("Tax years in pilot:")
    display(pilot["tax_year"].value_counts(dropna=False).sort_index().to_frame("filings"))
    print("Unknown raw responses:")
    display(pilot.loc[pilot.fraud_label.isna(), "raw_diversion_response"].value_counts(dropna=False).head(15))
    print("Missing EIN:", pilot.ein.isna().sum())
    print("Duplicate EIN + tax_year:", pilot.duplicated(["ein","tax_year"]).sum())
    if pilot.fraud_label.notna().mean() < .5:
        print("WARNING: Under 50% of full Form 990 sample has a recognized disclosure. Review Step 5 field names before scaling.")

In [ ]:
# Rates use filings with a recognized Yes/No response as the denominator.
positive_count = int((pilot["fraud_label"] == 1).sum())
negative_count = int((pilot["fraud_label"] == 0).sum())
missing_count = int(pilot["fraud_label"].isna().sum())
known_count = positive_count + negative_count

print("Positive cases:", positive_count)
print("Negative cases:", negative_count)
print("Missing labels:", missing_count)
if known_count:
    print("Positive rate among known responses:", f"{positive_count / known_count:.3%}")
else:
    print("Positive rate among known responses: N/A (no recognized responses)")


In [ ]:
# Inspect organizations reporting significant diversion of assets

positive_cases = pilot[pilot["fraud_label"] == 1].copy()

display(
    positive_cases.head(20)[
        [
            "ein",
            "organization_name",
            "tax_year",
            "publication_year",
            "source_xml",
            "raw_diversion_response"
        ]
    ]
)

In [ ]:
# Verify a positive disclosure against its source XML

if positive_cases.empty:
    print("No positive disclosures found in this sample.")
else:
    positive_filename = positive_cases.iloc[0]["source_xml"]

    with zipfile.ZipFile(zip_path) as z:
        with z.open(positive_filename) as f:
            root = ET.parse(f).getroot()

        for element in root.iter():
            if "diversion" in local_name(element.tag).lower():
                print(
                    local_name(element.tag),
                    "=",
                    repr(element.text)
                )

## 8 — Save pilot outputs and reproducibility metadata
Only save after checking Step 7. `data/staging/` is Git-ignored; a small JSON summary is written to `reports/`. No raw XML or organization-level sensitive details are committed.

In [ ]:
if records:
    # This output is a pilot audit, not the existing model score dataset.
    # Keep the whole pilot (including older tax years) for auditing; filter later by actual tax_year.
    output = STAGING / f"form990_fraud_label_pilot_pub{PILOT_PUBLICATION_YEAR}.parquet"
    pilot.to_parquet(output, index=False)
    summary = {
        "source_page": IRS_PAGE,
        "source_archive": selected,
        "publication_year": PILOT_PUBLICATION_YEAR,
        "target_tax_years": sorted(YEARS),
        "max_xml_files": MAX_XML_FILES,
        "xml_sampled": min(MAX_XML_FILES, len(xml_members)),
        "form990_parsed": len(pilot),
        "label_yes": int((pilot.fraud_label == 1).sum()),
        "label_no": int((pilot.fraud_label == 0).sum()),
        "label_missing": int(pilot.fraud_label.isna().sum()),
        "parse_errors": len(errors),
        "label_definition": "Self-reported significant diversion of assets; not confirmed fraud",
        "label_tags_considered": sorted(LABEL_TAGS),
    }
    report_path = REPORTS / "00_fraud_label_pilot_summary.json"
    report_path.write_text(json.dumps(summary, indent=2))
    print("Saved:", output)
    print("Saved:", report_path)
    display(pd.DataFrame([summary]).T.rename(columns={0:"value"}))